In [ ]:
#!/usr/bin/env python3
"""
Reference implementation of the maximal-clique decomposition algorithm
(Algorithm 1 of "An Algorithm for the Maximal Clique Problem of Graphs
using Algebraic Techniques").

The point of this file is that it realises the complexity bound proved in
Theorem 8.3: monomials are stored as integer bit masks over the forward
neighbourhood N^r(v_i), so that intersection, union and inclusion of
monomials are single machine operations, and the maximality tests are the
ones used in the proof.  This is the only difference from a naive symbolic
implementation, and it is worth several orders of magnitude.

Usage
-----
    python3 mcd_reference.py edgelist.txt          # run and report statistics
    python3 mcd_reference.py edgelist.txt --check  # also verify against networkx

The edge list is a whitespace-separated file, one edge per line; lines
beginning with '#' or '%' are ignored.  Vertices may be arbitrary tokens.

Outputs the table columns used in Section 9 of the paper:
    |V|, |E|, d, omega, lambda, #omega, |F|, time.
"""
def main():
    # -------------------------------------------------
    # Specify your edge list file here
    # -------------------------------------------------
    path = "Data Sets//facebook_combined.txt"      # <-- Change this to your file name/path
    check = False                # Set True to compare with NetworkX

    adj = read_edgelist(path)
    n = len(adj)
    m = sum(len(a) for a in adj.values()) // 2

    t0 = time.perf_counter()
    cliques, d, size_F = maximal_clique_decomposition(adj)
    elapsed = time.perf_counter() - t0

    omega = max((len(c) for c in cliques), default=0)
    n_omega = sum(1 for c in cliques if len(c) == omega)
    biggest = min((sorted(c) for c in cliques if len(c) == omega), default=[])

    print("file            :", path)
    print("|V|             :", n)
    print("|E|             :", m)
    print("degeneracy d    :", d)
    print("omega           :", omega)
    print("lambda = |mu(g)|:", len(cliques))
    print("#omega          :", n_omega)
    print("|F|             : %d   (|F|/lambda = %.3f)"
          % (size_F, size_F / len(cliques) if cliques else 0))
    print("a maximum clique:", biggest)
    print("time (s)        : %.4f" % elapsed)

    if check:
        try:
            import networkx as nx
        except ImportError:
            print("\n[NetworkX not installed]")
            return

        G = nx.Graph()
        G.add_nodes_from(adj)

        for u in adj:
            for v in adj[u]:
                G.add_edge(u, v)

        t1 = time.perf_counter()
        ref = {frozenset(c) for c in nx.find_cliques(G)}
        t_ref = time.perf_counter() - t1

        ours = {frozenset(c) for c in cliques}

        print("\nNetworkX find_cliques : %d cliques in %.4f s"
              % (len(ref), t_ref))
        print("Agreement             :",
              "OK" if ours == ref else "MISMATCH")


if __name__ == "__main__":
    main()